In [15]:
#!/usr/bin/env python3
import sympy as sp
from IPython.display import display, Math

# ---------- symbols ----------
q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)          # q1=waist yaw, q2=hip pitch, q3=knee pitch
L1, L2, L3 = sp.symbols('L1 L2 L3', positive=True, real=True)
hx, hy, hz = sp.symbols('hx hy hz', real=True)          # hip offset (base->hip)

# ---------- homogeneous transforms ----------
def Tx(a):
    return sp.Matrix([[1,0,0,a],
                      [0,1,0,0],
                      [0,0,1,0],
                      [0,0,0,1]])

def Ty(a):
    return sp.Matrix([[1,0,0,0],
                      [0,1,0,a],
                      [0,0,1,0],
                      [0,0,0,1]])

def Tz(a):
    return sp.Matrix([[1,0,0,0],
                      [0,1,0,0],
                      [0,0,1,a],
                      [0,0,0,1]])

def Rx(th):
    c, s = sp.cos(th), sp.sin(th)
    return sp.Matrix([[1,0,0,0],
                      [0,c,-s,0],
                      [0,s, c,0],
                      [0,0,0,1]])

def Ry(th):
    c, s = sp.cos(th), sp.sin(th)
    return sp.Matrix([[ c,0, s,0],
                      [ 0,1, 0,0],
                      [-s,0, c,0],
                      [ 0,0, 0,1]])

def Rz(th):
    c, s = sp.cos(th), sp.sin(th)
    return sp.Matrix([[c,-s,0,0],
                      [s, c,0,0],
                      [0, 0,1,0],
                      [0, 0,0,1]])

# ---------- RR leg chain ----------
# Base -> Hip offset
TBH = Tx(hx) * Ty(hy) * Tz(hz)

T_align = Ry(sp.pi/2)

# Hip yaw (waist) about Z
TH0 = Rz(q1)

# Hip pitch about Y, then link L1 along X
T01 = Ry(q2) * Tx(L1)

# Knee pitch about Y, then link (L2+L3) along X to foot/wheel-center
T12 = Ry(q3) * Tx(L2 + L3)

# Full transform: base -> foot
TBF = sp.simplify(TBH * T_align * TH0 * T01 * T12)
# TBF = sp.simplify(TBH * TH0 * T01 * T12)

# ---------- position ----------
x, y, z = sp.simplify(TBF[0,3]), sp.simplify(TBF[1,3]), sp.simplify(TBF[2,3])
p = sp.Matrix([x, y, z])
display(Math(r"[x,y,z]^T = " + sp.latex(p)))

# ---------- Jacobian (3x3) ----------
J = sp.simplify(p.jacobian([q1, q2, q3]))
display(Math(r"\mathbf{J}=" + sp.latex(J)))

# ---------- orientation (optional) ----------
R = TBF[:3,:3]
roll  = sp.atan2(R[2,1], R[2,2])
pitch = sp.atan2(-R[2,0], sp.sqrt(R[2,1]**2 + R[2,2]**2))
yaw   = sp.atan2(R[1,0], R[0,0])
pose_expr = sp.Matrix([x, y, z, roll, pitch, yaw])
# display(Math(r"[x,y,z,\phi,\theta,\psi]^T=" + sp.latex(pose_expr)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Inverse Kinematic

In [2]:
import sympy as sp
from IPython.display import display, Math

# ==========================================
# 1. กำหนดตัวแปร (Symbols)
# ==========================================
# พารามิเตอร์โครงสร้างของขา
x1, z1, z2 = sp.symbols('x_1 z_1 z_2')
L1, L2, beta = sp.symbols('L_1 L_2 \beta')

# เป้าหมายระยะและพิกัดที่ผ่านการ Bound แล้ว
x_des, z_des = sp.symbols('x_{des} z_{des}')
L_req, L_safe = sp.symbols('L_{req} L_{safe}')
x_safe, z_safe = sp.symbols('x_{safe} z_{safe}')

# มุมข้อต่อและตัวแปรคำนวณ
C_knee, q_knee, q_hip = sp.symbols('C_{knee} q_{knee} q_{hip}')
X_w, Z_w = sp.symbols('X_w Z_w')

# ==========================================
# 2. สร้างสมการ (Equations)
# ==========================================
# --- โครงสร้างขา (Leg Structure) ---
eq_L1 = sp.Eq(L1, sp.sqrt(x1**2 + z1**2))
eq_L2 = sp.Eq(L2, z2)
eq_beta = sp.Eq(beta, sp.atan2(-x1, z1))

# --- การหาระยะ (Distance Calculation) ---
eq_Lreq = sp.Eq(L_req, sp.sqrt(x_des**2 + z_des**2))

# --- มุมเข่า (Knee Angle - กฎของโคไซน์) ---
eq_Cknee = sp.Eq(C_knee, (L_safe**2 - L1**2 - L2**2) / (2 * L1 * L2))
eq_qknee = sp.Eq(q_knee, beta + sp.acos(C_knee))

# --- มุมสะโพก (Hip Angle - Robust Trigonometry) ---
eq_Xw = sp.Eq(X_w, x1 - L2 * sp.sin(q_knee))
eq_Zw = sp.Eq(Z_w, z1 + L2 * sp.cos(q_knee))

# คำนวณ sin, cos ของ hip
denominator = X_w**2 + Z_w**2
cos_q_hip = (X_w * x_safe + Z_w * z_safe) / denominator
sin_q_hip = (X_w * z_safe - Z_w * x_safe) / denominator

eq_qhip = sp.Eq(q_hip, sp.atan2(sin_q_hip, cos_q_hip))

# ==========================================
# 3. แสดงผล (Display)
# ==========================================
print("1. พารามิเตอร์โครงสร้างของขา (Leg Structural Parameters):")
display(Math(sp.latex(eq_L1)))
display(Math(sp.latex(eq_L2)))
display(Math(sp.latex(eq_beta)))

print("\n2. ระยะกระจัดที่ต้องการ (Required Distance):")
display(Math(sp.latex(eq_Lreq)))

print("\n3. การคำนวณมุมเข่า (Knee Angle Calculation):")
display(Math(sp.latex(eq_Cknee)))
display(Math(sp.latex(eq_qknee)))

print("\n4. การคำนวณมุมสะโพก (Hip Angle Calculation):")
display(Math(sp.latex(eq_Xw)))
display(Math(sp.latex(eq_Zw)))
display(Math(sp.latex(eq_qhip)))

1. พารามิเตอร์โครงสร้างของขา (Leg Structural Parameters):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


2. ระยะกระจัดที่ต้องการ (Required Distance):


...

<IPython.core.display.Math object>


4. การคำนวณมุมสะโพก (Hip Angle Calculation):


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Control [Wheel]

LQR

In [3]:
import sympy as sp
from IPython.display import display, Math

# 1. กำหนดตัวแปรสัญลักษณ์ (Symbolic Variables)
# พารามิเตอร์ของระบบ
g, L, M_eff, r_w = sp.symbols('g L M_{eff} r_w', real=True, positive=True)
I_yy, I_zz = sp.symbols('I_{yy} I_{zz}', real=True, positive=True)
c_pitch, c_v, c_yaw = sp.symbols('c_{pitch} c_v c_{yaw}', real=True, positive=True)

# ตัวแปรสถานะ (State) และ ตัวแปรควบคุม (Input)
theta, theta_dot, v, psi_dot = sp.symbols(r'\theta \dot{\theta} v \dot{\psi}', real=True)
tau_pitch, tau_yaw = sp.symbols(r'\tau_{pitch} \tau_{yaw}', real=True)

# 2. สร้างเวกเตอร์ State (x) และ Input (u)
x = sp.Matrix([[theta], [theta_dot], [v], [psi_dot]])
u = sp.Matrix([[tau_pitch], [tau_yaw]])

# 3. สร้างเมทริกซ์ A และ B ตามสมการเชิงเส้นของคุณ
A = sp.Matrix([
    [0, 1, 0, 0],
    [g/L, -c_pitch/I_yy, 0, 0],
    [0, 0, -c_v, 0],
    [0, 0, 0, -c_yaw/I_zz]
])

B = sp.Matrix([
    [0, 0],
    [-1/(I_yy * r_w), 0],
    [1/(M_eff * r_w), 0],
    [0, 1/I_zz]
])

# ฟังก์ชันตัวช่วยสำหรับแสดงผลให้สวยงาม
def print_equation(name, matrix):
    display(Math(name + ' = ' + sp.latex(matrix)))

# 4. แสดงผลสมการ
display(Math(r'\textbf{1. State and Input Vectors}'))
print_equation('x', x)
print_equation('u', u)

display(Math(r'\textbf{2. System Dynamics Matrices}'))
print_equation('A', A)
print_equation('B', B)

# 5. แสดงสมการ State-Space รวม
display(Math(r'\textbf{3. State-Space Equation: } \dot{x} = Ax + Bu'))
x_dot = sp.Matrix([[sp.Symbol(r'\dot{\theta}')], 
                   [sp.Symbol(r'\ddot{\theta}')], 
                   [sp.Symbol(r'\dot{v}')], 
                   [sp.Symbol(r'\ddot{\psi}')]])

# โชว์รูปสมการเต็มๆ
eq_full = sp.Eq(x_dot, sp.MatAdd(sp.MatMul(A, x), sp.MatMul(B, u)))
display(Math(sp.latex(eq_full)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

WBC

In [9]:
import sympy as sp
from IPython.display import display, Math

# ==========================================
# 1. กำหนดตัวแปรสัญลักษณ์ (Symbolic Variables)
# ==========================================

# 1.1 ตัวแปรสปริงที่เข่า (Knee Torsion Spring)
tau_spring, sgn, k_spring, b_spring = sp.symbols(r'\tau_{spring} sign k_{spring} b_{spring}', real=True)
q_knee, q_0, q_dot_knee = sp.symbols(r'q_{knee} q_0 \dot{q}_{knee}', real=True)

# 1.2 ตัวแปร Task-Space PD Control
u_task, K_p, K_d = sp.symbols(r'u_{task} K_p K_d', real=True)
x_ref, x, x_dot_ref, x_dot = sp.symbols(r'x_{ref} x \dot{x}_{ref} \dot{x}', real=True)

# 1.3 ตัวแปร Optimization (ID-QP)
J_total = sp.symbols(r'J_{total}', real=True)
w_z, w_roll, w_pitch, w_yaw, w_posture = sp.symbols(r'w_{height} w_{roll} w_{pitch} w_{yaw} w_{posture}', real=True, positive=True)

# [แก้ไข] ใช้ sp.Symbol (ไม่มี s) ประกาศทีละตัวเพื่อบังคับไม่ให้ SymPy แอบหั่นคำ
e_z = sp.Symbol(r'\|e_{z}\|^2', real=True)
e_roll = sp.Symbol(r'\|e_{roll}\|^2', real=True)
e_pitch = sp.Symbol(r'\|e_{pitch}\|^2', real=True)
e_yaw = sp.Symbol(r'\|e_{yaw}\|^2', real=True)
e_posture = sp.Symbol(r'\|e_{posture}\|^2', real=True)

M = sp.Symbol(r'M(q)')
q_ddot = sp.Symbol(r'\ddot{q}')
h = sp.Symbol(r'h(q,\dot{q})')  # ตัวปัญหาคือลูกน้ำตรงนี้! ตอนนี้ปลอดภัยแล้ว
S = sp.Symbol(r'S^T')
tau = sp.Symbol(r'\tau')
J_c = sp.Symbol(r'J_c^T')
F_c = sp.Symbol(r'F_c')

# ==========================================
# 2. สร้างสมการ (Equations)
# ==========================================

def print_equation(title, eq):
    display(Math(r'\textbf{' + title + r'}'))
    if isinstance(eq, str):
        display(Math(eq))
    else:
        display(Math(sp.latex(eq)))
    print("-" * 40)

display(Math(r'\Large\textbf{Mathematical Formulation for Ambula WBC Node}'))
print("=" * 60)

# --- สมการที่ 1: Knee Torsion Spring (สปริงจำลองที่เข่า) ---
eq_spring = sp.Eq(tau_spring, sgn * (-k_spring * (q_knee - q_0) - b_spring * q_dot_knee))
print_equation('1. Simulated Knee Torsion Spring Dynamics', eq_spring)


# --- สมการที่ 2: Task-Space PD Control Law ---
eq_pd = sp.Eq(u_task, K_p * (x_ref - x) + K_d * (x_dot_ref - x_dot))
print_equation('2. General Task-Space PD Control', eq_pd)


# --- สมการที่ 3: Inverse Dynamics Equation (Constraints) ---
LHS_dyn = M * q_ddot + h
RHS_dyn = S * tau + J_c * F_c
eq_dyn = sp.Eq(LHS_dyn, RHS_dyn)
print_equation('3. Rigid Body Dynamics Constraint (Equality)', eq_dyn)


# --- สมการที่ 4: Torque Limits (Constraints) ---
eq_tau_limit = r'\tau_{min} \leq \tau \leq \tau_{max}'
print_equation('4. Actuator Torque Limits (Inequality / Override)', eq_tau_limit)


# --- สมการที่ 5: WBC Objective Function (Cost Function) ---
cost_expr = (w_z * e_z) + (w_roll * e_roll) + (w_pitch * e_pitch) + (w_yaw * e_yaw) + (w_posture * e_posture)
eq_cost = sp.Eq(J_total, cost_expr)
print_equation('5. QP Objective Function (Weighted Task Prioritization)', eq_cost)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

----------------------------------------


<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

----------------------------------------


<IPython.core.display.Math object>

<IPython.core.display.Math object>

----------------------------------------
